### Utils file containing reussable functions

In [11]:
!git clone https://github.com/huggingface/lerobot.git
!apt-get install ffmpeg
!cd lerobot && pip install -e ".[train, dataset, smolvla]"

fatal: destination path 'lerobot' already exists and is not an empty directory.
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:6.1.1-3ubuntu5).
0 upgraded, 0 newly installed, 0 to remove and 33 not upgraded.
Obtaining file:///content/lerobot
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for lerobot (pyproject.toml) ... done
  Created wheel for lerobot: filename=lerobot-0.6.2-0.editable-py3-none-any.whl size=14936 sha256=c5e8dd9fb5488c20346c0460cf8dd99db941d957251fb4c63335fa1438bcedc9
  Stored in directory: /tmp/pip-ephem-wheel-cache-g0p7wdu5/wheels/f5/22/8e/9cd8447aef1caddc7ed6147de4cd2043f66eb6e84c52525987
Successfully built lerobot
  Attempting uninstall: lerobot
    Found existing installation: lerobot 0.6.2
    Uni

inspired from : https://colab.research.google.com/github/huggingface/notebooks/blob/main/lerobot/training-smolvla.ipynb#scrollTo=FMqvtVcThrdN

In [2]:
import os, getpass
os.environ["WANDB_API_KEY"] = getpass.getpass("Paste your W&B API key: ")

import wandb
wandb.login() 

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: tjga-98 (tjga-98-epfl) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [3]:
import os, getpass
from huggingface_hub import login

token = getpass.getpass("Paste your HF token: ")
login(token=token)              # saves it for this machine, so later `!` commands find it
os.environ["HF_TOKEN"] = token

### Post-training SmolVLA using PEFT

In [4]:
!pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [8]:
# !lerobot-train \
#  --resume=true \
#  --config_path=TomasAnderegg/pill_sorting_post_trained

!lerobot-train \
 --policy.path=lerobot/smolvla_base \
 --policy.output_features=null \
 --policy.input_features=null \
 --policy.optimizer_lr=1e-3 \
 --policy.device=cuda \
 --policy.scheduler_decay_lr=1e-4 \
 --policy.repo_id=TomasAnderegg/pill_sorting_post_trained_a100 \
 --num_workers=10  \
 --save_freq=3500 \
 --log_freq=100 \
 --accelerator.mixed_precision=bf16 \
 --dataset.repo_id=TomasAnderegg/so101_pill_sorting_merged_v2 \
 --steps=10000 \
 --batch_size=64 \
 --peft.method_type=LORA \
 --peft.r=64 \
 --peft.lora_alpha=64 \
 --wandb.enable=true \
 --wandb.project=pill_smolvla \
 --wandb.entity=tjga-98-epfl \
 --save_checkpoint_to_hub=true 

config.json: 100% 2.30k/2.30k [00:00<00:00, 1.27MB/s]
INFO 2026-10-07 08:28:12 ot_train.py:490 {'accelerator': {'activation_checkpointing': {'mode': <ActivationCheckpointingMode.NONE: 'none'>},
                 'compile': {'backend': 'inductor',
                             'enabled': False,
                             'mode': None,
                             'regional': True},
                 'ddp': {'find_unused_parameters': True,
                         'gradient_as_bucket_view': False,
                         'static_graph': False},
                 'fsdp': {'cpu_offload': False,
                          'ignored_modules': None,
                          'min_num_params': None,
                          'reshard_after_forward': True,
                          'wrap_modules': None},
                 'grad_scaler': {'backoff_factor': 0.5,
                                 'growth_factor': 2.0,
                                 'growth_interval': 2000,
                           

In [6]:
 !nvidia-smi


Wed Oct  7 08:18:41 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   33C    P0             56W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [7]:
!nproc

12


Calculation of the number of parameters used with LoRa:
1. Check the target used for LoRa: "(model\\.vlm_with_expert\\.lm_expert\\..*\\.(q|v)_proj|model\\.(state_proj|action_in_proj|action_out_proj|action_time_mlp_in|action_time_mlp_out))"
2. For each of them extract the features_in and features_out
3. Since LoRa adds 2 matrices A with size r x features_in and B with size features_out x r, we can calculate the total number for each adapted target.
4. Add everything

In [ ]:
from lerobot.policies.smolvla.modeling_smolvla import SmolVLAPolicy

policy = SmolVLAPolicy.from_pretrained("lerobot/smolvla_base")
print(policy)

Loading  HuggingFaceTB/SmolVLM2-500M-Video-Instruct weights ...


Loading weights:   0%|          | 0/489 [00:00<?, ?it/s]

Reducing the number of VLM layers to 16 ...
SmolVLAPolicy(
  (model): VLAFlowMatching(
    (vlm_with_expert): SmolVLMWithExpertModel(
      (vlm): SmolVLMForConditionalGeneration(
        (model): SmolVLMModel(
          (vision_model): SmolVLMVisionTransformer(
            (embeddings): SmolVLMVisionEmbeddings(
              (patch_embedding): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16), padding=valid)
              (position_embedding): Embedding(1024, 768)
            )
            (encoder): SmolVLMEncoder(
              (layers): ModuleList(
                (0-11): 12 x SmolVLMEncoderLayer(
                  (self_attn): SmolVLMVisionAttention(
                    (k_proj): Linear(in_features=768, out_features=768, bias=True)
                    (v_proj): Linear(in_features=768, out_features=768, bias=True)
                    (q_proj): Linear(in_features=768, out_features=768, bias=True)
                    (out_proj): Linear(in_features=768, out_features=768, bias=True)